# Alternativas — zeros, assimetria e taxas limitadas

PPML em hectares admite valores não inteiros e zeros. Logit fracional admite taxas 0 e 1, sem inventar número de ensaios. Ambos usam FE de ano e bioma, não FE municipal. A hipótese de média condicional é forte; convergência não assegura especificação correta. Covariância cluster município/ano e sensibilidade municipal. Não há aptidão pecuária validada: a exposição agrícola para pastagem examina associação cruzada, não capacidade de produzir carne.

**Inferência:** se a matriz two-way tiver autovalor negativo, aplica-se projeção espectral (autovalores negativos iguais a zero). SE bruto permanece exportado para auditoria; a correção é determinada pelo diagnóstico, nunca por significância. Sensibilidade municipal/HAC também é salva. A projeção não resolve poucos clusters nem ausência de identificação.

In [1]:
from utilitarios import *
from IPython.display import display
d = ler()
print(f"{len(d)} linhas, {d.id.nunique()} municípios, {d.ano.min()}–{d.ano.max()}")

3384 linhas, 141 municípios, 2001–2024


## 1. Conversão em hectares com PPML — dois destinos

In [2]:
for uso in ['agricultura','pastagem']:
    ident='Q01_'+uso; y='ha_'+uso
    s=amostra(d,[y,'expo_soja','expo_milho','z_a_soja','z_a_milho','bioma'],ident)
    formula=f'{y} ~ expo_soja + expo_milho + z_a_soja + z_a_milho + C(bioma) + C(ano)'
    r,cov,cm=estimar(formula,s,'PPML')
    pred=r.predict(s)
    assert np.isfinite(pred).all()
    display(exportar(ident,'Conversão em hectares com PPML',formula,s,r,cov,'PPML','ano+bioma',y,cm=cm,alerta='sem FE municipal; coeficientes em escala da ligação; verificar efeitos marginais'))
    display(marginais(r,cov,s,ident,'PPML'))
    diag={'id':ident,'min_pred':pred.min(),'max_pred':pred.max(),'media_observada':s[y].mean(),'media_predita':pred.mean(),'rmse':np.sqrt(np.mean((s[y]-pred)**2)),'zeros':s[y].eq(0).sum(),'max_abs_coef':np.max(np.abs(r.params)),'iteracoes':r.fit_history.get('iteration'),'convergiu':r.converged}
    pd.DataFrame([diag]).to_csv(PASTA/f'diagnostico_{ident}.csv',index=False)
    display(pd.DataFrame([diag]))

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,Q01_agricultura,Intercept,5.977048,0.219043,5.523923,6.430173,5.050531e-19,2772,sem FE municipal; coeficientes em escala da li...
26,Q01_agricultura,expo_soja,0.062691,0.030116,0.000391,0.124991,4.869445e-02,2772,sem FE municipal; coeficientes em escala da li...
27,Q01_agricultura,expo_milho,0.075160,0.052105,-0.032627,0.182947,1.626472e-01,2772,sem FE municipal; coeficientes em escala da li...
28,Q01_agricultura,z_a_soja,-0.504131,0.160991,-0.837166,-0.171096,4.684883e-03,2772,sem FE municipal; coeficientes em escala da li...
29,Q01_agricultura,z_a_milho,-0.102161,0.174494,-0.463128,0.258807,5.639328e-01,2772,sem FE municipal; coeficientes em escala da li...


,id,preco,ame,se,ic95_inf,ic95_sup,unidade,limite
0,Q01_agricultura,soja,-10.586139,7.512400,-26.126722,4.954445,ha por 1 DP preço,componente diferencial preço×aptidão; efeito c...
1,Q01_agricultura,milho,-9.673250,7.545899,-25.283132,5.936631,ha por 1 DP preço,componente diferencial preço×aptidão; efeito c...


,id,min_pred,max_pred,media_observada,media_predita,rmse,zeros,max_abs_coef,iteracoes,convergiu
0,Q01_agricultura,1.779063,9033.919182,297.697312,297.697312,942.235146,265,5.977048,8,True


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,Q01_pastagem,Intercept,8.474897,0.123563,8.219287,8.730507,4.160942e-28,2772,sem FE municipal; coeficientes em escala da li...
26,Q01_pastagem,expo_soja,0.071542,0.048499,-0.028785,0.171869,1.537350e-01,2772,sem FE municipal; coeficientes em escala da li...
27,Q01_pastagem,expo_milho,0.027442,0.020727,-0.015436,0.070319,1.985264e-01,2772,sem FE municipal; coeficientes em escala da li...
28,Q01_pastagem,z_a_soja,-0.391452,0.175756,-0.755032,-0.027873,3.600230e-02,2772,sem FE municipal; coeficientes em escala da li...
29,Q01_pastagem,z_a_milho,0.195868,0.124462,-0.061601,0.453336,1.292086e-01,2772,sem FE municipal; coeficientes em escala da li...


,id,preco,ame,se,ic95_inf,ic95_sup,unidade,limite
0,Q01_pastagem,soja,-78.957163,31.431643,-143.978470,-13.935855,ha por 1 DP preço,componente diferencial preço×aptidão; efeito c...
1,Q01_pastagem,milho,-13.059828,15.116822,-44.331357,18.211701,ha por 1 DP preço,componente diferencial preço×aptidão; efeito c...


,id,min_pred,max_pred,media_observada,media_predita,rmse,zeros,max_abs_coef,iteracoes,convergiu
0,Q01_pastagem,644.769612,27186.092969,3257.588132,3257.588132,4065.645361,0,8.474897,6,True


## 2. Taxa com logit fracional — dois destinos

In [3]:
for uso in ['agricultura','pastagem']:
    ident='F01_'+uso; y='taxa_'+uso
    s=amostra(d,[y,'expo_soja','expo_milho','z_a_soja','z_a_milho','bioma'],ident)
    formula=f'{y} ~ expo_soja + expo_milho + z_a_soja + z_a_milho + C(bioma) + C(ano)'
    r,cov,cm=estimar(formula,s,'Fracional')
    pred=r.predict(s)
    assert np.isfinite(pred).all()
    display(exportar(ident,'Taxa com logit fracional',formula,s,r,cov,'Fracional','ano+bioma',y,cm=cm,alerta='sem FE municipal; coeficientes em escala da ligação; verificar efeitos marginais'))
    display(marginais(r,cov,s,ident,'Fracional'))
    diag={'id':ident,'min_pred':pred.min(),'max_pred':pred.max(),'media_observada':s[y].mean(),'media_predita':pred.mean(),'rmse':np.sqrt(np.mean((s[y]-pred)**2)),'zeros':s[y].eq(0).sum(),'max_abs_coef':np.max(np.abs(r.params)),'iteracoes':r.fit_history.get('iteration'),'convergiu':r.converged}
    pd.DataFrame([diag]).to_csv(PASTA/f'diagnostico_{ident}.csv',index=False)
    display(pd.DataFrame([diag]))

,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,F01_agricultura,Intercept,-6.386987,0.243388,-6.890474,-5.883500,1.207191e-18,2772,sem FE municipal; coeficientes em escala da li...
26,F01_agricultura,expo_soja,0.045964,0.062551,-0.083433,0.175360,4.698761e-01,2772,sem FE municipal; coeficientes em escala da li...
27,F01_agricultura,expo_milho,0.050537,0.084393,-0.124044,0.225117,5.551428e-01,2772,sem FE municipal; coeficientes em escala da li...
28,F01_agricultura,z_a_soja,-0.215620,0.174789,-0.577199,0.145959,2.298108e-01,2772,sem FE municipal; coeficientes em escala da li...
29,F01_agricultura,z_a_milho,-0.358738,0.217761,-0.809210,0.091734,1.130722e-01,2772,sem FE municipal; coeficientes em escala da li...


,id,preco,ame,se,ic95_inf,ic95_sup,unidade,limite
0,F01_agricultura,soja,-0.000023,0.000037,-0.000099,0.000053,fração por 1 DP preço,componente diferencial preço×aptidão; efeito c...
1,F01_agricultura,milho,-0.000024,0.000040,-0.000106,0.000058,fração por 1 DP preço,componente diferencial preço×aptidão; efeito c...


,id,min_pred,max_pred,media_observada,media_predita,rmse,zeros,max_abs_coef,iteracoes,convergiu
0,F01_agricultura,0.000002,0.020668,0.001122,0.001122,0.003516,265,6.386987,14,True


,id,termo,coef,se,ic95_inf,ic95_sup,p,N,alertas
0,F01_pastagem,Intercept,-3.654263,0.088770,-3.837897,-3.470629,4.756340e-23,2772,sem FE municipal; coeficientes em escala da li...
26,F01_pastagem,expo_soja,0.072318,0.033066,0.003915,0.140721,3.916555e-02,2772,sem FE municipal; coeficientes em escala da li...
27,F01_pastagem,expo_milho,0.057151,0.028277,-0.001344,0.115646,5.505704e-02,2772,sem FE municipal; coeficientes em escala da li...
28,F01_pastagem,z_a_soja,0.178501,0.078636,0.015830,0.341172,3.289551e-02,2772,sem FE municipal; coeficientes em escala da li...
29,F01_pastagem,z_a_milho,0.146145,0.095779,-0.051990,0.344279,1.406825e-01,2772,sem FE municipal; coeficientes em escala da li...


,id,preco,ame,se,ic95_inf,ic95_sup,unidade,limite
0,F01_pastagem,soja,0.000029,0.000045,-0.000064,0.000122,fração por 1 DP preço,componente diferencial preço×aptidão; efeito c...
1,F01_pastagem,milho,-0.000037,0.000030,-0.000099,0.000025,fração por 1 DP preço,componente diferencial preço×aptidão; efeito c...


,id,min_pred,max_pred,media_observada,media_predita,rmse,zeros,max_abs_coef,iteracoes,convergiu
0,F01_pastagem,0.002721,0.041294,0.011915,0.011915,0.012303,0,3.654263,8,True
